# Premier League — Modelo de Previsão de Resultados

**Objetivo:** treinar e avaliar um `HistGradientBoostingClassifier` para prever o resultado de partidas da Premier League em três classes: *HomeWin*, *Draw*, *AwayWin*.

**Premissa:** todas as features já estão calculadas e prontas nos parquets gerados pelo `notebook_data.ipynb`. Este notebook não realiza nenhuma transformação de dados — apenas carrega, treina e avalia.

| Arquivo | Conteúdo |
|---------|----------|
| `dados/feature_matrix_train.parquet` | Treino — seasons 2005/06 → 2021/22 |
| `dados/feature_matrix_test.parquet` | Teste — seasons 2022/23 → 2025/26 |

> **Baseline ingênuo (sempre HomeWin):** 45,6 %

## Seção 1 — Imports

In [ ]:
import os
os.environ["MPLBACKEND"] = "agg"

import difflib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("Imports OK.")

## Seção 2 — Load Data

In [ ]:
train = pd.read_parquet("dados/feature_matrix_train.parquet")
test  = pd.read_parquet("dados/feature_matrix_test.parquet")

print(f"Treino : {train.shape}  |  seasons {train['season'].min()}–{train['season'].max()}")
print(f"Teste  : {test.shape}  |  seasons {test['season'].min()}–{test['season'].max()}")
train.head(3)

## Seção 3 — Definir X e y

In [ ]:
NON_FEATURE_COLS = [
    # Identificadores e metadados
    "Date", "Season", "season", "home_team", "away_team",
    # Alvo e resultado bruto
    "result", "FTR",
    # Scores brutos — leakage (não disponíveis antes do apito)
    "home_score", "away_score", "HTHG", "HTAG", "HTR",
    # Estatísticas brutas do jogo — leakage
    "HS", "AS", "HST", "AST", "HF", "AF",
    "HC", "AC", "HY", "AY", "HR", "AR", "Referee",
    # Odds brutas — já normalizadas como implied_*_prob
    "B365H", "B365D", "B365A",
    # Elo — redundante com as odds implícitas; remover desta lista para reativar
    "elo_home", "elo_away", "elo_diff",
]

FEATURE_COLS = [c for c in train.columns if c not in NON_FEATURE_COLS]

X_train = train[FEATURE_COLS]
y_train = train["result"]
X_test  = test[FEATURE_COLS]
y_test  = test["result"]

print(f"Total de features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

## Seção 4 — Recency Weighting

Jogos mais recentes recebem peso maior via decaimento exponencial por temporada. Com `α = 0.10`, a temporada mais recente pesa ~e^(17×0.1) ≈ 5.5× mais que a primeira.

In [ ]:
season_weight = np.exp(0.10 * (train["season"] - train["season"].min()))
sample_weight = (season_weight / season_weight.mean()).values

print(f"Peso mínimo (season {train['season'].min()}): {sample_weight.min():.3f}")
print(f"Peso máximo (season {train['season'].max()}): {sample_weight.max():.3f}")

## Seção 5 — Treino

`HistGradientBoostingClassifier` com early stopping interno: reserva 10% dos dados de treino (os mais recentes, pois estão ordenados por data) como validação para parar automaticamente quando não há melhora por 30 iterações.

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=1000,
    max_depth=5,
    min_samples_leaf=12,
    l2_regularization=0.2,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
)
hgb.fit(X_train, y_train, sample_weight=sample_weight)
print(f"Iterações efetivas (early stopping): {hgb.n_iter_}")

## Seção 6 — Cross-Validation

`TimeSeriesSplit(n_splits=5)` respeita a ordem temporal: cada fold usa apenas dados passados para treinar e dados futuros para validar. Os pesos de recência são aplicados em cada fold de treino separadamente.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_acc, cv_f1 = [], []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    w_tr = sample_weight[tr_idx]

    m = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=500, max_depth=5,
        min_samples_leaf=12, l2_regularization=0.2,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=30, random_state=42,
    )
    m.fit(X_tr, y_tr, sample_weight=w_tr)
    preds = m.predict(X_val)
    cv_acc.append(accuracy_score(y_val, preds))
    cv_f1.append(f1_score(y_val, preds, average="macro"))
    print(f"  Fold {fold}: acc={cv_acc[-1]:.3f}  macro-F1={cv_f1[-1]:.3f}")

print(f"\nCV Accuracy : {np.mean(cv_acc):.3f} ± {np.std(cv_acc):.3f}")
print(f"CV Macro-F1 : {np.mean(cv_f1):.3f} ± {np.std(cv_f1):.3f}")

## Seção 7 — Avaliação no Teste

O conjunto de teste é tocado **uma única vez**, aqui. Qualquer ajuste de hiperparâmetros baseado neste resultado seria data leakage.

In [ ]:
y_pred = hgb.predict(X_test)

print(classification_report(
    y_test, y_pred,
    labels=["HomeWin", "Draw", "AwayWin"],
    digits=3,
))

cm = confusion_matrix(y_test, y_pred, labels=["HomeWin", "Draw", "AwayWin"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["HomeWin", "Draw", "AwayWin"],
    yticklabels=["HomeWin", "Draw", "AwayWin"],
    ax=ax,
)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("Confusion Matrix — Conjunto de Teste")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

## Seção 8 — Baseline Comparison

In [ ]:
acc       = accuracy_score(y_test, y_pred)
macro_f1  = f1_score(y_test, y_pred, average="macro")
naive_acc = 0.456

print(f"{'Método':<30} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 52)
print(f"{'Naive (sempre HomeWin)':<30} {naive_acc:>10.1%} {'—':>10}")
print(f"{'HistGradientBoosting':<30} {acc:>10.1%} {macro_f1:>10.3f}")
print(f"\nGanho sobre baseline: +{(acc - naive_acc):.1%}")

## Seção 9 — `predict_match()`

Função de inferência ad-hoc: recebe dois nomes de times e retorna a predição do modelo usando as features mais recentes disponíveis de cada time no dataset combinado (treino + teste).

In [ ]:
all_df = pd.concat([train, test], ignore_index=True).sort_values("Date").reset_index(drop=True)


def predict_match(home_team_name: str, away_team_name: str) -> str:
    all_teams = sorted(set(all_df["home_team"]) | set(all_df["away_team"]))

    def _resolve(name: str) -> str:
        if name in all_teams:
            return name
        suggestions = difflib.get_close_matches(name, all_teams, n=3, cutoff=0.6)
        suffix = f" Você quis dizer: {suggestions}?" if suggestions else ""
        raise ValueError(f"Time '{name}' não encontrado.{suffix}")

    h = _resolve(home_team_name)
    a = _resolve(away_team_name)

    def _last_row_and_role(team: str):
        candidates = []
        home_rows = all_df[all_df["home_team"] == team]
        away_rows = all_df[all_df["away_team"] == team]
        if len(home_rows):
            r = home_rows.loc[home_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "home"))
        if len(away_rows):
            r = away_rows.loc[away_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "away"))
        if not candidates:
            raise ValueError(f"Sem dados históricos para '{team}'")
        _, last, role = max(candidates, key=lambda x: x[0])
        return last, role

    h_row, h_role = _last_row_and_role(h)
    a_row, a_role = _last_row_and_role(a)

    # Stats de forma por time (independente do papel em cada partida)
    team_stats = [
        "goals_scored_last5", "goals_conceded_last5",
        "wins_last5", "draws_last5", "losses_last5",
        "goal_diff_last5", "points_last5",
        "shots_on_target_last5", "corners_last5",
        "win_pct_season", "draw_pct_season", "loss_pct_season",
        "ppg_season", "goal_eff",
    ]

    feat = {}
    for s in team_stats:
        feat[f"home_{s}"] = h_row.get(f"{h_role}_{s}", 0.0)
        feat[f"away_{s}"] = a_row.get(f"{a_role}_{s}", 0.0)

    # Features específicas de local (última aparição naquele papel)
    h_home = all_df[all_df["home_team"] == h].sort_values("Date")
    a_away = all_df[all_df["away_team"] == a].sort_values("Date")
    feat["home_wins_at_home_last5"] = float(h_home["home_wins_at_home_last5"].iloc[-1]) if len(h_home) else 0.0
    feat["away_wins_away_last5"]    = float(a_away["away_wins_away_last5"].iloc[-1])    if len(a_away) else 0.0

    # Elo (pré-partida do último jogo do time)
    feat["elo_home"] = float(h_row[f"elo_{h_role}"])
    feat["elo_away"] = float(a_row[f"elo_{a_role}"])
    feat["elo_diff"] = feat["elo_home"] - feat["elo_away"]

    # Probs implícitas (sem odds ao vivo → média global como prior)
    feat["implied_home_prob"] = float(all_df["implied_home_prob"].mean())
    feat["implied_draw_prob"] = float(all_df["implied_draw_prob"].mean())
    feat["implied_away_prob"] = float(all_df["implied_away_prob"].mean())

    # Taxas históricas
    feat["home_team_hist_wr"] = float(h_home["home_team_hist_wr"].iloc[-1]) if len(h_home) else float(all_df["home_team_hist_wr"].mean())
    feat["away_team_hist_wr"] = float(a_away["away_team_hist_wr"].iloc[-1]) if len(a_away) else float(all_df["away_team_hist_wr"].mean())

    # Fase da temporada: meio de temporada como fallback
    feat["round_norm"] = 0.5

    # Vantagem relativa (recalculada)
    feat["goals_scored_adv"]   = feat["home_goals_scored_last5"]    - feat["away_goals_scored_last5"]
    feat["goals_conceded_adv"] = feat["away_goals_conceded_last5"]  - feat["home_goals_conceded_last5"]
    feat["points_adv"]         = feat["home_points_last5"]          - feat["away_points_last5"]
    feat["goal_diff_adv"]      = feat["home_goal_diff_last5"]       - feat["away_goal_diff_last5"]
    feat["win_pct_adv"]        = feat["home_win_pct_season"]        - feat["away_win_pct_season"]
    feat["shots_adv"]          = feat["home_shots_on_target_last5"] - feat["away_shots_on_target_last5"]
    feat["ppg_adv"]            = feat["home_ppg_season"]            - feat["away_ppg_season"]
    feat["goal_eff_adv"]       = feat["home_goal_eff"]              - feat["away_goal_eff"]

    X_inf = pd.DataFrame([feat])[FEATURE_COLS]
    pred  = hgb.predict(X_inf)[0]
    proba = dict(zip(hgb.classes_, hgb.predict_proba(X_inf)[0].round(3)))

    print(f"{h} (casa)  vs  {a} (fora)")
    print(f"Predição    : {pred}")
    print(f"Probabilidades: {proba}")
    return pred

In [ ]:
# Teste com exemplo real
_ = predict_match("Arsenal", "Chelsea")
print()

# Teste com nome inválido — deve sugerir o nome correto
try:
    predict_match("Arsnel", "Chelsea")
except ValueError as e:
    print(f"Erro esperado: {e}")

## Seção 10 — Interpretação dos Resultados

### Resultados obtidos

| Métrica | Valor |
|---------|------:|
| **Accuracy (teste)** | **51,8 %** |
| Baseline ingênuo (sempre HomeWin) | 45,6 % |
| Ganho sobre baseline | **+6,2 pp** |
| **Macro-F1 (teste)** | **0,405** |
| CV Accuracy (TimeSeriesSplit × 5) | 52,1 % ± 1,5 % |
| CV Macro-F1 | 0,432 ± 0,021 |

---

### Recall por classe

| Classe | Recall | Interpretação |
|--------|-------:|---------------|
| HomeWin | 0,746 | O modelo identifica bem a maioria das vitórias do mandante |
| AwayWin | 0,565 | Desempenho razoável nas vitórias do visitante |
| Draw | 0,036 | Empates são muito difíceis de prever — classe com menos sinal |

O baixo recall em *Draw* é esperado e universal em modelos de futebol: empates são intrinsecamente ruidosos, sem correlação forte com features pré-jogo. A maioria dos modelos da literatura simplesmente aprende a não predizer esta classe.

---

### Contexto e teto realista

O **teto realista para previsão de futebol com dados abertos** (sem dados de tracking, escalações ao vivo ou odds em tempo real) é estimado entre **54–58 % de accuracy** nos melhores trabalhos da literatura. Modelos baseados exclusivamente em form e Elo tipicamente atingem **51–54 %**.

O resultado de **51,8 %** está dentro do esperado para este conjunto de features. Para avançar além deste patamar, as principais alavancas são:

1. **Odds ao vivo** como feature de inferência (não apenas históricas)
2. **Dados de escalação e lesões** — ausência de titulares é o maior preditor individual
3. **Expected Goals (xG)** — mais informativo que gols marcados para estimar forma real
4. **Calibração das probabilidades** — `CalibratedClassifierCV` para melhorar as saídas de `predict_proba`

## Seção 11 — Predição Seletiva por Confiança

Em vez de forçar uma predição para cada partida, o modelo **abstém** quando a probabilidade máxima não supera um limiar de confiança `θ`.

```
confiança = max(P(HomeWin), P(Draw), P(AwayWin))
se confiança ≥ θ → retorna predição
se confiança <  θ → abstém
```

**Trade-off:** quanto maior o `θ`, menos partidas passam o filtro (*cobertura* cai), mas a *precisão* nas que passam aumenta. O objetivo aqui não é accuracy global — é ter alta confiança nas poucas partidas que o modelo escolhe responder.

In [ ]:
# Confiança = probabilidade máxima do modelo principal (treinado em 2005–2021)
proba_test_full = hgb.predict_proba(X_test)
classes_order   = list(hgb.classes_)

confidence    = proba_test_full.max(axis=1)
predicted_cls = np.array([classes_order[i] for i in proba_test_full.argmax(axis=1)])
correct       = predicted_cls == y_test.values

# Curva cobertura × precisão para limiares de 0.45 a 0.90
thresholds  = np.arange(0.45, 0.91, 0.01)
coverages, precisions, counts = [], [], []

for θ in thresholds:
    mask = confidence >= θ
    n    = mask.sum()
    counts.append(n)
    coverages.append(n / len(confidence))
    precisions.append(correct[mask].mean() if n > 0 else np.nan)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(thresholds, precisions, color="#2196F3", lw=2, label="Precisão (acertos / filtradas)")
ax2.plot(thresholds, coverages,  color="#FF9800", lw=2, linestyle="--", label="Cobertura (% partidas)")

ax1.axhline(0.456, color="gray", linestyle=":", lw=1, label="Baseline naive (45.6%)")
ax1.axvline(0.85,  color="red",  linestyle="--", lw=1, label="θ = 0.85")

ax1.set_xlabel("Limiar de confiança (θ)")
ax1.set_ylabel("Precisão", color="#2196F3")
ax2.set_ylabel("Cobertura", color="#FF9800")
ax1.set_ylim(0.40, 1.05)
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
ax1.set_title("Predição seletiva — Precisão vs. Cobertura (conjunto de teste)")
plt.tight_layout()
plt.savefig("selective_prediction_curve.png", dpi=120)
plt.show()

# Tabela de referência
print(f"\n{'θ':>6}  {'Partidas':>9}  {'Cobertura':>10}  {'Precisão':>10}")
print("-" * 42)
for θ in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    mask = confidence >= θ
    n    = mask.sum()
    prec = correct[mask].mean() if n > 0 else float("nan")
    cov  = n / len(confidence)
    print(f"{θ:>6.2f}  {n:>9}  {cov:>10.1%}  {prec:>10.1%}")

In [ ]:
def predict_match_selective(
    home_team_name: str,
    away_team_name: str,
    confidence_threshold: float = 0.70,
) -> dict | None:
    """
    Retorna predição apenas se a confiança (max proba) >= threshold.
    Caso contrário retorna None — o modelo abstém.
    """
    all_teams = sorted(set(all_df["home_team"]) | set(all_df["away_team"]))

    def _resolve(name: str) -> str:
        if name in all_teams:
            return name
        suggestions = difflib.get_close_matches(name, all_teams, n=3, cutoff=0.6)
        suffix = f" Você quis dizer: {suggestions}?" if suggestions else ""
        raise ValueError(f"Time '{name}' não encontrado.{suffix}")

    h = _resolve(home_team_name)
    a = _resolve(away_team_name)

    def _last_row_and_role(team):
        candidates = []
        home_rows = all_df[all_df["home_team"] == team]
        away_rows = all_df[all_df["away_team"] == team]
        if len(home_rows):
            r = home_rows.loc[home_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "home"))
        if len(away_rows):
            r = away_rows.loc[away_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "away"))
        _, last, role = max(candidates, key=lambda x: x[0])
        return last, role

    h_row, h_role = _last_row_and_role(h)
    a_row, a_role = _last_row_and_role(a)

    team_stats = [
        "goals_scored_last5", "goals_conceded_last5",
        "wins_last5", "draws_last5", "losses_last5",
        "goal_diff_last5", "points_last5",
        "shots_on_target_last5", "corners_last5",
        "win_pct_season", "draw_pct_season", "loss_pct_season",
        "ppg_season", "goal_eff",
    ]

    feat = {}
    for s in team_stats:
        feat[f"home_{s}"] = h_row.get(f"{h_role}_{s}", 0.0)
        feat[f"away_{s}"] = a_row.get(f"{a_role}_{s}", 0.0)

    h_home = all_df[all_df["home_team"] == h].sort_values("Date")
    a_away = all_df[all_df["away_team"] == a].sort_values("Date")
    feat["home_wins_at_home_last5"] = float(h_home["home_wins_at_home_last5"].iloc[-1]) if len(h_home) else 0.0
    feat["away_wins_away_last5"]    = float(a_away["away_wins_away_last5"].iloc[-1])    if len(a_away) else 0.0
    feat["elo_home"] = float(h_row[f"elo_{h_role}"])
    feat["elo_away"] = float(a_row[f"elo_{a_role}"])
    feat["elo_diff"] = feat["elo_home"] - feat["elo_away"]
    feat["implied_home_prob"] = float(all_df["implied_home_prob"].mean())
    feat["implied_draw_prob"] = float(all_df["implied_draw_prob"].mean())
    feat["implied_away_prob"] = float(all_df["implied_away_prob"].mean())
    feat["home_team_hist_wr"] = float(h_home["home_team_hist_wr"].iloc[-1]) if len(h_home) else float(all_df["home_team_hist_wr"].mean())
    feat["away_team_hist_wr"] = float(a_away["away_team_hist_wr"].iloc[-1]) if len(a_away) else float(all_df["away_team_hist_wr"].mean())
    feat["round_norm"]        = 0.5
    feat["goals_scored_adv"]   = feat["home_goals_scored_last5"]    - feat["away_goals_scored_last5"]
    feat["goals_conceded_adv"] = feat["away_goals_conceded_last5"]  - feat["home_goals_conceded_last5"]
    feat["points_adv"]         = feat["home_points_last5"]          - feat["away_points_last5"]
    feat["goal_diff_adv"]      = feat["home_goal_diff_last5"]       - feat["away_goal_diff_last5"]
    feat["win_pct_adv"]        = feat["home_win_pct_season"]        - feat["away_win_pct_season"]
    feat["shots_adv"]          = feat["home_shots_on_target_last5"] - feat["away_shots_on_target_last5"]
    feat["ppg_adv"]            = feat["home_ppg_season"]            - feat["away_ppg_season"]
    feat["goal_eff_adv"]       = feat["home_goal_eff"]              - feat["away_goal_eff"]

    X_inf = pd.DataFrame([feat])[FEATURE_COLS]
    proba = hgb.predict_proba(X_inf)
    conf  = float(proba.max())
    pred  = hgb.classes_[int(proba.argmax())]
    proba_dict = dict(zip(hgb.classes_, proba[0].round(3)))

    if conf < confidence_threshold:
        print(f"{h} vs {a}  →  ABSTÉM  (confiança {conf:.1%} < θ={confidence_threshold:.0%})")
        return None

    print(f"{h} (casa) vs {a} (fora)")
    print(f"  Predição   : {pred}")
    print(f"  Confiança  : {conf:.1%}")
    print(f"  Probabilidades: {proba_dict}")
    return {"prediction": pred, "confidence": conf, "proba": proba_dict}


# Demonstração
jogos = [
    ("Man City",  "Sheffield United"),
    ("Arsenal",   "Chelsea"),
    ("Liverpool", "Everton"),
    ("Burnley",   "Man City"),
]

print("=== Predição seletiva (θ = 70%) ===\n")
for home, away in jogos:
    predict_match_selective(home, away, confidence_threshold=0.70)
    print()